In [7]:
import os, re, json, io, zipfile, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

# ── 경로 설정 ──────────────────────────────────────────────
TEAM_DIR   = "/content/drive/MyDrive/1팀 공유 문서/김태윤"
DRIVE_BASE = "/content/drive/MyDrive/여기에_경로/dataset"  # ← 수정
OUTPUT_IMG_56 = os.path.join(DRIVE_BASE, "dataset_56_masked", "images")
OUTPUT_ANN_56 = os.path.join(DRIVE_BASE, "dataset_56_masked", "annotations")
OUTPUT_IMG_74 = os.path.join(DRIVE_BASE, "dataset_74_masked", "images")
OUTPUT_ANN_74 = os.path.join(DRIVE_BASE, "dataset_74_masked", "annotations")
for d in [OUTPUT_IMG_56, OUTPUT_ANN_56, OUTPUT_IMG_74, OUTPUT_ANN_74]:
    os.makedirs(d, exist_ok=True)

ZIP_NUMS     = [1, 3, 4, 5, 6, 7, 8]
MASK_PADDING = 10
CENTER_SIZE  = 100

# ── 타겟 클래스 ───────────────────────────────────────────
TARGET_X = {
    1900, 2483, 3351, 3483, 3544, 3743, 3832, 4543,
    12081, 12247, 12778, 13395, 13900, 16232, 16262, 16548,
    16551, 16688, 18147, 18357, 19232, 19552, 19607, 19861,
    20014, 20238, 20877, 21325, 21771, 22074, 22347, 22362,
    24850, 25367, 25438, 25469, 27733, 27777, 27926, 27993,
    28763, 29345, 29451, 29667, 30308, 31863, 31885, 32310,
    33009, 33208, 33880, 34597, 35206, 36637, 38162, 41768,
}
TARGET_N = {
    27653, 23223, 6563, 6192, 31705, 18110, 10221, 21026,
    5094, 44199, 23203, 10224, 33878, 22627, 29871,
    4378, 5886, 12420,
}
TARGET_74 = TARGET_X | TARGET_N



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# ── 유틸 함수 ─────────────────────────────────────────────
def extract_group_key(filename):
    m = re.match(r'(K-[\d-]+?)_0_2', os.path.basename(filename))
    return m.group(1) if m else None

def extract_kcode(folder_name):
    m = re.search(r'K-0*(\d+)', folder_name)
    return int(m.group(1)) if m else None

def sample_bg_color(img_array, size=CENTER_SIZE):
    H, W  = img_array.shape[:2]
    cy, cx= H//2, W//2
    half  = size // 2
    patch = img_array[max(0,cy-half):cy+half, max(0,cx-half):cx+half]
    return tuple(patch.mean(axis=(0,1)).astype(np.uint8))

def mask_bbox(img_array, bbox, bg_color, padding=MASK_PADDING):
    if len(bbox) < 4:
        return img_array
    x, y, w, h = [int(v) for v in bbox]
    H, W = img_array.shape[:2]
    x1 = max(0, x-padding);   y1 = max(0, y-padding)
    x2 = min(W, x+w+padding); y2 = min(H, y+h+padding)
    img_array[y1:y2, x1:x2] = bg_color
    return img_array

def build_coco_categories(target_set):
    return [{"id": k, "name": str(k), "supercategory": "pill"}
            for k in sorted(target_set)]



In [9]:
# ── Step 1: TL 파싱 → 그룹별 수집 ───────────────────────
print("=" * 55)
print("Step 1. TL 파싱")
print("=" * 55)

all_groups = defaultdict(dict)  # group_key → {fn: [(kcode, bbox)]}

for num in ZIP_NUMS:
    tl_path = os.path.join(TEAM_DIR, f"TL_{num}_조합.zip")
    if not os.path.exists(tl_path):
        print(f"  [SKIP] TL_{num} 없음")
        continue
    print(f"  TL_{num}_조합.zip 파싱 중...")
    with zipfile.ZipFile(tl_path, 'r') as z:
        json_files = [f for f in z.namelist() if f.endswith('.json')]
        for jf in json_files:
            try:
                data      = json.loads(z.read(jf))
                img       = data["images"][0]
                ann       = data["annotations"][0]
                fn        = img["file_name"]
                folder    = jf.split('/')[1]
                kcode     = extract_kcode(folder)
                if kcode is None: continue
                if len(ann.get("bbox", [])) < 4: continue
                group_key = extract_group_key(fn)
                if group_key is None: continue
                if fn not in all_groups[group_key]:
                    all_groups[group_key][fn] = []
                all_groups[group_key][fn].append((kcode, ann["bbox"]))
            except: pass

print(f"전체 그룹 수: {len(all_groups)}")



Step 1. TL 파싱
  TL_1_조합.zip 파싱 중...
  TL_3_조합.zip 파싱 중...
  TL_4_조합.zip 파싱 중...
  TL_5_조합.zip 파싱 중...
  TL_6_조합.zip 파싱 중...
  TL_7_조합.zip 파싱 중...
  TL_8_조합.zip 파싱 중...
전체 그룹 수: 3500


In [10]:
# ── Step 2: 그룹별 랜덤 1장 선택 ─────────────────────────
print("\n" + "=" * 55)
print("Step 2. 그룹별 랜덤 1장 선택")
print("=" * 55)

selected = {}
for group_key, fn_dict in all_groups.items():
    chosen_fn = random.choice(list(fn_dict.keys()))
    selected[chosen_fn] = fn_dict[chosen_fn]

print(f"선택 후 이미지 수: {len(selected)}장")

# ── Step 3: 필터링 ────────────────────────────────────────
print("\n" + "=" * 55)
print("Step 3. 필터링")
print("=" * 55)

filtered_56 = {
    fn: pills for fn, pills in selected.items()
    if any(k in TARGET_X for k, _ in pills)
}
filtered_74 = {
    fn: pills for fn, pills in selected.items()
    if any(k in TARGET_74 for k, _ in pills)
}

print(f"56종 포함 이미지: {len(filtered_56)}장")
print(f"74종 포함 이미지: {len(filtered_74)}장")




Step 2. 그룹별 랜덤 1장 선택
선택 후 이미지 수: 3500장

Step 3. 필터링
56종 포함 이미지: 2615장
74종 포함 이미지: 2749장


In [ ]:
# ── Step 4: TS 마스킹 + 저장 (56종 / 74종 동시 처리) ────
print("\n" + "=" * 55)
print("Step 4. 마스킹 + 저장")
print("=" * 55)

coco_56 = {"images": [], "annotations": [],
            "categories": build_coco_categories(TARGET_X)}
coco_74 = {"images": [], "annotations": [],
            "categories": build_coco_categories(TARGET_74)}

img_id_56 = ann_id_56 = 1
img_id_74 = ann_id_74 = 1
saved_56 = saved_74 = 0
viz_56, viz_74 = [], []

for num in ZIP_NUMS:
    ts_path = os.path.join(TEAM_DIR, f"TS_{num}_조합.zip")
    if not os.path.exists(ts_path):
        print(f"  [SKIP] TS_{num} 없음")
        continue
    print(f"  TS_{num}_조합.zip 처리 중...")

    with zipfile.ZipFile(ts_path, 'r') as z:
        img_map = {
            os.path.basename(f): f
            for f in z.namelist()
            if f.lower().endswith('.png') and 'index' not in f
        }

        # ── 56종 처리 ─────────────────────────────────────
        for fn, pills in filtered_56.items():
            img_path = img_map.get(fn)
            if not img_path: continue

            img_bytes = z.read(img_path)
            img_pil   = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            img_arr   = np.array(img_pil)
            H, W      = img_arr.shape[:2]
            bg_color  = sample_bg_color(img_arr)

            for kcode, bbox in pills:
                if kcode not in TARGET_X:
                    img_arr = mask_bbox(img_arr, bbox, bg_color)

            Image.fromarray(img_arr).save(os.path.join(OUTPUT_IMG_56, fn))
            coco_56["images"].append(
                {"id": img_id_56, "file_name": fn, "width": W, "height": H})
            for kcode, bbox in pills:
                if kcode in TARGET_X:
                    coco_56["annotations"].append({
                        "id": ann_id_56, "image_id": img_id_56,
                        "category_id": kcode,
                        "bbox": [float(v) for v in bbox],
                        "area": float(bbox[2]*bbox[3]), "iscrowd": 0,
                    })
                    ann_id_56 += 1
            img_id_56 += 1; saved_56 += 1
            if len(viz_56) < 10:
                viz_56.append((img_arr.copy(), pills, fn, TARGET_X))

        # ── 74종 처리 ─────────────────────────────────────
        for fn, pills in filtered_74.items():
            img_path = img_map.get(fn)
            if not img_path: continue

            img_bytes = z.read(img_path)
            img_pil   = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            img_arr   = np.array(img_pil)
            H, W      = img_arr.shape[:2]
            bg_color  = sample_bg_color(img_arr)

            for kcode, bbox in pills:
                if kcode not in TARGET_74:
                    img_arr = mask_bbox(img_arr, bbox, bg_color)

            Image.fromarray(img_arr).save(os.path.join(OUTPUT_IMG_74, fn))
            coco_74["images"].append(
                {"id": img_id_74, "file_name": fn, "width": W, "height": H})
            for kcode, bbox in pills:
                if kcode in TARGET_74:
                    coco_74["annotations"].append({
                        "id": ann_id_74, "image_id": img_id_74,
                        "category_id": kcode,
                        "bbox": [float(v) for v in bbox],
                        "area": float(bbox[2]*bbox[3]), "iscrowd": 0,
                    })
                    ann_id_74 += 1
            img_id_74 += 1; saved_74 += 1
            if len(viz_74) < 10:
                viz_74.append((img_arr.copy(), pills, fn, TARGET_74))

print(f"56종 저장: {saved_56}장 / {ann_id_56-1}개 bbox")
print(f"74종 저장: {saved_74}장 / {ann_id_74-1}개 bbox")




Step 4. 마스킹 + 저장
  TS_1_조합.zip 처리 중...
  TS_3_조합.zip 처리 중...


In [ ]:
# ── COCO JSON 저장 ────────────────────────────────────────
for coco, ann_dir in [(coco_56, OUTPUT_ANN_56), (coco_74, OUTPUT_ANN_74)]:
    p = os.path.join(ann_dir, "_annotations.coco.json")
    with open(p, "w") as f:
        json.dump(coco, f, ensure_ascii=False)
print("COCO JSON 저장 완료")



In [ ]:
# ── 샘플 시각화 ──────────────────────────────────────────
def visualize_samples(viz_buffer, title):
    if not viz_buffer: return
    n = len(viz_buffer)
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))
    axes = np.array(axes).flatten()
    for i, (img_arr, pills, fn, target) in enumerate(viz_buffer):
        ax = axes[i]
        ax.imshow(img_arr)
        for kcode, bbox in pills:
            if kcode in target:
                x, y, w, h = bbox
                color = "lime" if kcode in TARGET_X else "cyan"
                rect  = patches.Rectangle((x,y), w, h,
                                           linewidth=2, edgecolor=color,
                                           facecolor="none")
                ax.add_patch(rect)
                ax.text(x, y-3, str(kcode), fontsize=6, color=color)
        ax.set_title(fn[:25], fontsize=6)
        ax.axis("off")
    for i in range(len(viz_buffer), len(axes)):
        axes[i].axis("off")
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

print("\n=== 56종 샘플 (초록=X종) ===")
visualize_samples(viz_56, f"dataset_56_masked (padding={MASK_PADDING}px)")

print("\n=== 74종 샘플 (초록=X종 / 하늘=N종) ===")
visualize_samples(viz_74, f"dataset_74_masked (padding={MASK_PADDING}px)")

print(f"\nMASK_PADDING 조정 필요 시 값 변경 후 재실행하세요. (현재: {MASK_PADDING}px)")